# Four-DOF LEGO EV3 Manipulator
Kinematics, Cartesian tracking, redundancy resolution and spherical-obstacle avoidance. This notebook preserves the original simulation algorithm and numeric settings. Historical simulation metrics are documented in the README; a new run produces separate results. No hardware is accessed.


## Dependencies
NumPy, Matplotlib and SymPy are sufficient for the simulation. Interactive sliders depend on the selected Matplotlib backend.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.widgets import Slider
from mpl_toolkits.mplot3d import Axes3D
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['font.size'] = 11
import sympy as sp
print('SymPy version:', sp.__version__)


## Geometry and joint limits
Four revolute joints plus a fixed auxiliary transform. All calculations use metres and radians; display angles use degrees. The fixed transform is not an additional joint.


In [ ]:
l1 = 0.08
l2 = 0.0
l3 = 0.0
l3_ = 0.0
l4 = 0.085
d1 = 0.0
d2 = 0.0
d3 = 0.045
d3_ = 0.11
d4 = 0.0
alpha1 = -np.pi / 2
alpha2 = np.pi / 2
alpha3 = -np.pi / 2
alpha3_ = 0.0
alpha4 = 0.0
q0 = np.array([0.0, 0.0, 0.0, 0.0])
q_min = np.radians(np.array([-120, -20, -124, -180]))
q_max = np.radians(np.array([90, 35, 45, 180]))
qdot_max = np.radians(np.array([40, 40, 40, 40]))
print('limits joint [deg]:')
for i in range(4):
    print(f'  q{i + 1}: [{np.degrees(q_min[i]):.1f}°, {np.degrees(q_max[i]):.1f}°]  → range {np.degrees(q_max[i] - q_min[i]):.1f}°')


## Forward kinematics
Multiply the base transform and five DH transforms in chain order. The base is translated 0.14 m vertically. Return every frame and link endpoint for Jacobian and clearance calculations.


In [ ]:
def DH_computation(d, a, alpha, theta):
    T = np.array([[np.cos(theta), -np.sin(theta) * np.cos(alpha), np.sin(theta) * np.sin(alpha), a * np.cos(theta)], [np.sin(theta), np.cos(theta) * np.cos(alpha), -np.cos(theta) * np.sin(alpha), a * np.sin(theta)], [0.0, np.sin(alpha), np.cos(alpha), d], [0.0, 0.0, 0.0, 1.0]])
    return T

def forward_kinematics(q):
    Tb0 = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0.14], [0, 0, 0, 1]])
    T01 = DH_computation(d1, l1, alpha1, q[0])
    T12 = DH_computation(d2, l2, alpha2, q[1])
    T23 = DH_computation(d3, l3, alpha3, q[2])
    T23_ = DH_computation(d3_, l3_, alpha3_, 0.0)
    T34 = DH_computation(d4, l4, alpha4, q[3])
    Tb1 = Tb0 @ T01
    Tb2 = Tb1 @ T12
    Tb3 = Tb2 @ T23
    Tb3_f = Tb3 @ T23_
    Tb4 = Tb3_f @ T34
    p0 = Tb0[:3, 3]
    p1 = Tb1[:3, 3]
    p2 = Tb2[:3, 3]
    p3 = Tb3[:3, 3]
    p3_f = Tb3_f[:3, 3]
    p4 = Tb4[:3, 3]
    base_to_tb0 = Tb0[:3, 3]
    norm = np.linalg.norm(base_to_tb0)
    mid_point = base_to_tb0 / norm * 0.025 if norm > 0 else np.array([0, 0, 1]) * 0.025
    return ((Tb0, Tb1, Tb2, Tb3, Tb3_f, Tb4), (p0, p1, p2, p3, p3_f, p4), mid_point)

def getEndEffector3D(q):
    _, (_, _, _, _, _, p4), _ = forward_kinematics(q)
    return p4.copy()

def getP3f3D(q):
    _, (_, _, _, _, p3_f, _), _ = forward_kinematics(q)
    return p3_f.copy()

def getMidFictitiousLink3D(q):
    _, (_, _, _, p3, p3_f, _), _ = forward_kinematics(q)
    return 0.5 * (p3 + p3_f)

def getAllRobotPoints(q):
    _, (p0, p1, p2, p3, p3_f, p4), _ = forward_kinematics(q)
    origin = np.zeros(3)
    return [(origin, p0), (p0, p1), (p1, p2), (p2, p3), (p3, p3_f), (p3_f, p4)]

def getAllRobotPointsFlat(q):
    _, (p0, p1, p2, p3, p3_f, p4), _ = forward_kinematics(q)
    origin = np.zeros(3)
    return np.array([origin, p0, p1, p2, p3, p3_f, p4])

def saturateJoints(q):
    return np.clip(q, q_min, q_max)

def saturateQdot(qdot):
    return np.clip(qdot, -qdot_max, qdot_max)
_, (p0, p1, p2, p3, p3_f, p4), _ = forward_kinematics(q0)
print('forward kinematics in q0 = [0, 0, 0, 0]:')
print(f'  p1   = {p1}')
print(f'  p2   = {p2}')
print(f'  p3   = {p3}')
print(f'  p3_f = {p3_f}')
print(f'  p4   = {p4}  ← end-effector')


### Robot visualisation
Frames, links and the spherical obstacle are displayed in the base coordinate system.


In [ ]:
def draw_frame(ax, T, scale=0.03, label_prefix='', fontsize=7):
    origin = T[:3, 3]
    for axis, color, label in zip(T[:3, :3].T, ['red', 'green', 'blue'], ['X', 'Y', 'Z']):
        ax.quiver(*origin, *axis, length=scale, color=color, linewidth=0.8, normalize=True)
        ax.text(*origin + axis * scale * 1.3, f'{label}{label_prefix}', color=color, fontsize=fontsize)

def plotSphere(ax, center, radius, color='gray', alpha=0.35):
    u = np.linspace(0, 2 * np.pi, 40)
    v = np.linspace(0, np.pi, 20)
    x = center[0] + radius * np.outer(np.cos(u), np.sin(v))
    y = center[1] + radius * np.outer(np.sin(u), np.sin(v))
    z = center[2] + radius * np.outer(np.ones_like(u), np.cos(v))
    ax.plot_surface(x, y, z, color=color, alpha=alpha, linewidth=0)

def drawRobot3D(ax, q, draw_frames=True, frame_scale=0.05):
    (Tb0, Tb1, Tb2, Tb3, Tb3_f, Tb4), (p0, p1, p2, p3, p3_f, p4), mid_point = forward_kinematics(q)
    origin = np.zeros(3)
    origin = np.zeros(3)
    ax.plot([0, p0[0]], [0, p0[1]], [0, p0[2]], 'gray', lw=2, ls='--')
    ax.plot(*zip(p0, p1), c='red', lw=3, label='Link 1')
    ax.plot(*zip(p1, p2), c='green', lw=3, label='Link 2')
    ax.plot(*zip(p2, p3), c='blue', lw=3, label='Link 3')
    ax.plot(*zip(p3, p3_f), c='cyan', lw=3, label='fixed auxiliary link')
    ax.plot(*zip(p3_f, p4), c='purple', lw=3, label='Link 4')
    ax.scatter(*np.array([mid_point, p1, p2, p3, p3_f, p4]).T, c='k', s=20)
    if draw_frames:
        T_world = np.eye(4)
        for T, label in zip([T_world, Tb0, Tb1, Tb2, Tb3, Tb3_f, Tb4], ['B', '0', '1', '2', '3', '3f', '4']):
            draw_frame(ax, T, scale=frame_scale, label_prefix=label)


In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')
plt.subplots_adjust(left=0.15, bottom=0.35)

def update_robot(val=None):
    ax.cla()
    q_slider = np.array([s.val for s in sliders])
    drawRobot3D(ax, q_slider, draw_frames=True)
    ax.set_xlim(-0.25, 0.25)
    ax.set_ylim(-0.25, 0.25)
    ax.set_zlim(-0.05, 0.35)
    ax.set_xlabel('X [m]')
    ax.set_ylabel('Y [m]')
    ax.set_zlabel('Z [m]')
    ax.set_title('Robot 4-DOF — view 3D')
    ax.legend(loc='upper left', fontsize=8)
    fig.canvas.draw_idle()
axcolor = 'lightgoldenrodyellow'
slider_axes = [plt.axes([0.15, 0.25 - i * 0.05, 0.7, 0.03], facecolor=axcolor) for i in range(4)]
sliders = [Slider(slider_axes[0], 'q1 [rad]', np.radians(-120), np.radians(90), valinit=0.0), Slider(slider_axes[1], 'q2 [rad]', np.radians(-20), np.radians(35), valinit=0.0), Slider(slider_axes[2], 'q3 [rad]', np.radians(-124), np.radians(45), valinit=0.0), Slider(slider_axes[3], 'q4 [rad]', np.radians(-180), np.radians(180), valinit=0.0)]
for s in sliders:
    s.on_changed(update_robot)
update_robot()
plt.show()


## Workspace and Cartesian waypoints
The original workspace samples 15 values per joint (50,625 configurations). A, B and C define a mostly horizontal trajectory, limiting demand on joint 2.


In [ ]:
A = np.array([0.165, 0.11, 0.185])
B = np.array([0.072, 0.184, 0.178])
C = np.array([-0.08, 0.185, 0.183])
dist_AB = np.linalg.norm(B - A)
dist_BC = np.linalg.norm(C - B)
print(f'A = {A}')
print(f'B = {B}')
print(f'C = {C}')
print(f'distance A→B = {dist_AB * 100:.1f} cm')
print(f'distance B→C = {dist_BC * 100:.1f} cm')
print(f'total distance = {(dist_AB + dist_BC) * 100:.1f} cm')
print('\ncomputing workspace (may take a few seconds)...')
q1_range = np.radians(np.linspace(-120, 90, 15))
q2_range = np.radians(np.linspace(-20, 35, 15))
q3_range = np.radians(np.linspace(-124, 45, 15))
q4_range = np.radians(np.linspace(-180, 180, 15))
workspace_pts = []
for q1 in q1_range:
    for q2 in q2_range:
        for q3 in q3_range:
            for q4 in q4_range:
                workspace_pts.append(getEndEffector3D([q1, q2, q3, q4]))
workspace_pts = np.array(workspace_pts)
print(f'Sampled points: {len(workspace_pts)}')


In [ ]:
fig1 = plt.figure(figsize=(8, 6))
ax1 = fig1.add_subplot(111, projection='3d')
ax1.scatter(workspace_pts[:, 0], workspace_pts[:, 1], workspace_pts[:, 2], s=1, c='skyblue', alpha=0.3)
for pt, label, color in [(A, 'A', 'black'), (B, 'B', 'red'), (C, 'C', 'magenta')]:
    ax1.scatter(*pt, c=color, s=60)
    ax1.text(pt[0], pt[1], pt[2] + 0.01, label, color='black')
ax1.plot([A[0], B[0]], [A[1], B[1]], [A[2], B[2]], 'k--', lw=2)
ax1.plot([B[0], C[0]], [B[1], C[1]], [B[2], C[2]], 'k--', lw=2)
ax1.set_xlabel('X [m]')
ax1.set_ylabel('Y [m]')
ax1.set_zlabel('Z [m]')
ax1.set_title('workspace 3D with trajectory A→B→C')
plt.tight_layout()
plt.show()
plt.figure(figsize=(7, 7))
plt.plot(workspace_pts[:, 0], workspace_pts[:, 1], '.', color='skyblue', alpha=0.2, ms=1.5)
for pt, label, color in [(A, 'A', 'ko'), (B, 'B', 'ro'), (C, 'C', 'mo')]:
    plt.plot(pt[0], pt[1], color, ms=8, label=label)
plt.plot([A[0], B[0]], [A[1], B[1]], 'k--', lw=2)
plt.plot([B[0], C[0]], [B[1], C[1]], 'k--', lw=2)
plt.xlabel('X [m]')
plt.ylabel('Y [m]')
plt.title('projection workspace — plane XY')
plt.axis('equal')
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()


## Cubic Cartesian planning
Each segment uses a cubic scalar displacement with zero endpoint velocities. A→B takes 8 s and B→C a further 8 s, with simulation step 0.02 s. Position and velocity are continuous at B; acceleration continuity is not imposed.


In [ ]:
t0_traj = 0.0
t1_traj = 8.0
t2_traj = 16.0
dt = 0.02

def cartesianPlanner(ti, tf, pi, pf, t):
    if t < ti:
        return (pi.copy(), np.zeros(3))
    if t <= tf:
        A_mat = np.array([[1, ti, ti ** 2, ti ** 3], [0, 1, 2 * ti, 3 * ti ** 2], [1, tf, tf ** 2, tf ** 3], [0, 1, 2 * tf, 3 * tf ** 2]])
        L = np.linalg.norm(pf - pi)
        if L < 1e-12:
            return (pi.copy(), np.zeros(3))
        b = np.array([0.0, 0.0, L, 0.0])
        coeff = np.linalg.solve(A_mat, b)
        s = coeff[0] + coeff[1] * t + coeff[2] * t ** 2 + coeff[3] * t ** 3
        sdot = coeff[1] + 2 * coeff[2] * t + 3 * coeff[3] * t ** 2
        direction = (pf - pi) / L
        return (pi + direction * s, direction * sdot)
    return (pf.copy(), np.zeros(3))

def desiredTrajectory(t):
    if t <= t1_traj:
        return cartesianPlanner(t0_traj, t1_traj, A, B, t)
    elif t <= t2_traj:
        return cartesianPlanner(t1_traj, t2_traj, B, C, t)
    return (C.copy(), np.zeros(3))
time_vec = np.arange(t0_traj, t2_traj + dt, dt)
Pd_preview = np.array([desiredTrajectory(t)[0] for t in time_vec])
Vd_preview = np.array([desiredTrajectory(t)[1] for t in time_vec])
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
labels = ['x [m]', 'y [m]', 'z [m]']
vlabels = ['vx [m/s]', 'vy [m/s]', 'vz [m/s]']
for i in range(3):
    axes[0, i].plot(time_vec, Pd_preview[:, i])
    axes[0, i].set_ylabel(labels[i])
    axes[0, i].grid()
    axes[1, i].plot(time_vec, Vd_preview[:, i], color='orange')
    axes[1, i].set_ylabel(vlabels[i])
    axes[1, i].set_xlabel('time [s]')
    axes[1, i].grid()
axes[0, 0].set_title('Desired position')
axes[1, 0].set_title('Desired velocity')
for ax in axes[0]:
    ax.axvline(t1_traj, color='gray', ls='--', lw=1)
for ax in axes[1]:
    ax.axvline(t1_traj, color='gray', ls='--', lw=1)
plt.suptitle('Planned Cartesian trajectory A→B→C (cubic law)')
plt.tight_layout()
plt.show()


## Linear Jacobians
The 3×4 end-effector Jacobian uses revolute-axis cross products. A separate midpoint Jacobian differentiates the fixed auxiliary link midpoint; its fourth column is zero.


In [ ]:
def Jacobian(q):
    (Tb0, Tb1, Tb2, Tb3, Tb3_f, Tb4), (p0, p1, p2, p3, p3_f, p4), _ = forward_kinematics(q)
    z0, o0 = (Tb0[:3, 2], p0)
    z1, o1 = (Tb1[:3, 2], p1)
    z2, o2 = (Tb2[:3, 2], p2)
    z3, o3 = (Tb3_f[:3, 2], p3_f)
    J = np.zeros((3, 4))
    J[:, 0] = np.cross(z0, p4 - o0)
    J[:, 1] = np.cross(z1, p4 - o1)
    J[:, 2] = np.cross(z2, p4 - o2)
    J[:, 3] = np.cross(z3, p4 - o3)
    return J

def JacobianMidFictitious(q):
    (Tb0, Tb1, Tb2, Tb3, Tb3_f, Tb4), (p0, p1, p2, p3, p3_f, p4), _ = forward_kinematics(q)
    pmid = 0.5 * (p3 + p3_f)
    z0, o0 = (Tb0[:3, 2], p0)
    z1, o1 = (Tb1[:3, 2], p1)
    z2, o2 = (Tb2[:3, 2], p2)
    Jp = np.zeros((3, 4))
    Jp[:, 0] = np.cross(z0, pmid - o0)
    Jp[:, 1] = np.cross(z1, pmid - o1)
    Jp[:, 2] = np.cross(z2, pmid - o2)
    Jp[:, 3] = np.zeros(3)
    return Jp


## Primary Cartesian tracking
Use qdot = pinv(J) @ (xdot_desired + K @ error), followed by velocity clipping, Euler integration and joint-position clipping. Error samples in the original loop compare the desired position at the current time with the previous-step configuration, rather than the post-update position.


In [ ]:
kgain = 10.0

def inverseKinematics(q, xd, xdot_d, kgain_val=1.0):
    J = Jacobian(q)
    x = getEndEffector3D(q)
    err = xd - x
    K = kgain_val * np.eye(3)
    qdot = np.linalg.pinv(J) @ (xdot_d + K @ err)
    qdot = saturateQdot(qdot)
    return (qdot, err)

def simulateTrajectoryNoFunctional(kgain_val=10.0):
    time = np.arange(t0_traj, t2_traj + dt, dt)
    N = len(time)
    Q = np.zeros((N, 4))
    Pd = np.zeros((N, 3))
    Preal = np.zeros((N, 3))
    error = np.zeros(N)
    Q[0, :] = q0
    Pd[0, :] = A
    Preal[0, :] = getEndEffector3D(q0)
    error[0] = np.linalg.norm(A - Preal[0, :])
    for i in range(1, N):
        t = time[i]
        pd, pdot = desiredTrajectory(t)
        qdot, err = inverseKinematics(Q[i - 1, :], pd, pdot, kgain_val=kgain_val)
        Q[i, :] = saturateJoints(Q[i - 1, :] + qdot * dt)
        Pd[i, :] = pd
        Preal[i, :] = getEndEffector3D(Q[i, :])
        error[i] = np.linalg.norm(err)
    return (time, Q, Pd, Preal, error)
time_no, Q_no, Pd_no, Preal_no, error_no = simulateTrajectoryNoFunctional(kgain_val=kgain)
print(f'mean error:  {np.mean(error_no) * 1000:.3f} mm')
print(f'maximum error: {np.max(error_no) * 1000:.3f} mm')
print(f'q2 max reached: {np.max(np.degrees(Q_no[:, 1])):.2f}°  (limit: 35°)')
print(f'q2 min reached: {np.min(np.degrees(Q_no[:, 1])):.2f}°  (limit: -20°)')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for i, (label, ax) in enumerate(zip(['x', 'y', 'z'], axes.flat)):
    ax.plot(time_no, Pd_no[:, i], 'r--', label='desired')
    ax.plot(time_no, Preal_no[:, i], 'b', label='encoder-derived')
    ax.axvline(t1_traj, color='gray', ls='--', lw=1)
    ax.set_ylabel(f'{label} [m]')
    ax.set_xlabel('time [s]')
    ax.legend()
    ax.grid()
axes[1, 1].plot(time_no, error_no * 1000, 'purple')
axes[1, 1].set_ylabel('error [mm]')
axes[1, 1].set_xlabel('time [s]')
axes[1, 1].set_title('tracking error')
axes[1, 1].grid()
plt.suptitle('results simulation without secondary task')
plt.tight_layout()
plt.show()
plt.figure(figsize=(11, 5))
q_min_deg = np.degrees(q_min)
q_max_deg = np.degrees(q_max)
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']
for i in range(4):
    line, = plt.plot(time_no, np.degrees(Q_no[:, i]), color=colors[i], label=f'q{i + 1}')
    plt.axhline(q_min_deg[i], ls='--', lw=1, color=colors[i], alpha=0.7)
    plt.axhline(q_max_deg[i], ls='--', lw=1, color=colors[i], alpha=0.7)
plt.xlabel('time [s]')
plt.ylabel('joint angle [deg]')
plt.title('trajectory joints with joint limits — without secondary task')
plt.legend(ncol=2)
plt.grid()
plt.tight_layout()
plt.show()


## Obstacle distance
The obstacle is a sphere. Local signed clearance is the midpoint-to-centre norm minus radius. Global clearance is the minimum sphere-to-segment distance across the chain. Positive centreline clearance is not a full volumetric collision guarantee.


In [ ]:
def pointSegmentClosestPoint(c, a, b):
    ab = b - a
    den = np.dot(ab, ab)
    t = 0.0 if den < 1e-12 else float(np.clip(np.dot(c - a, ab) / den, 0.0, 1.0))
    closest = a + t * ab
    return (closest, t, float(np.linalg.norm(c - closest)))
obstacle_center = np.array([-0.005, 0.062, 0.183])
obstacle_radius = 0.025
activation_distance = 0.06

def pointSegmentDistance(c, a, b):
    ab = b - a
    den = np.dot(ab, ab)
    if den < 1e-12:
        return np.linalg.norm(c - a)
    t = np.clip(np.dot(c - a, ab) / den, 0.0, 1.0)
    return np.linalg.norm(c - (a + t * ab))

def obstacleDistance(q):
    pmid = getMidFictitiousLink3D(q)
    return np.linalg.norm(pmid - obstacle_center) - obstacle_radius

def globalObstacleDistance(q):
    segments = getAllRobotPoints(q)
    dlist = [pointSegmentDistance(obstacle_center, a, b) - obstacle_radius for a, b in segments]
    return float(np.min(dlist))

def gradObstacleDistance(q):
    pmid = getMidFictitiousLink3D(q)
    delta = pmid - obstacle_center
    norm = np.linalg.norm(delta)
    if norm < 1e-12:
        return np.zeros(4)
    u = delta / norm
    Jmid = JacobianMidFictitious(q)
    return Jmid.T @ u
d_init = obstacleDistance(q0)
d_glob = globalObstacleDistance(q0)
print(f'distance signed midpoint fixed auxiliary link - sphere: {d_init * 100:.2f} cm')
print(f'global distance manipulator - sphere:             {d_glob * 100:.2f} cm')
print()
if d_init < 0:
    print('⚠ Collision in the initial configuration!')
elif d_init < activation_distance:
    print('✓ Within activation zone; secondary task initially active.')
else:
    print('✓ Outside activation zone in initial configuration.')


In [ ]:
dist_no_hist = np.array([obstacleDistance(Q_no[k, :]) for k in range(len(Q_no))])
dglob_no_hist = np.array([globalObstacleDistance(Q_no[k, :]) for k in range(len(Q_no))])
k_min = int(np.argmin(dist_no_hist))
d_min = dist_no_hist[k_min]
segs = getAllRobotPoints(Q_no[k_min, :])
dists = [pointSegmentDistance(obstacle_center, a, b) - obstacle_radius for a, b in segs]
seg_crit = int(np.argmin(dists))
p_close, _, _ = pointSegmentClosestPoint(obstacle_center, segs[seg_crit][0], segs[seg_crit][1])
print('=== Minimum-clearance analysis without secondary task ===')
print(f'Critical instant: k = {k_min}, t = {time_no[k_min]:.2f} s')
print(f'minimum distance (midpoint fixed auxiliary link): {d_min * 100:.3f} cm')
if d_min < 0:
    print('\n→ without secondary task collision occurs in this test case.')
else:
    print(f'\n→ without secondary task minimum distance positive ({d_min * 100:.2f} cm): no collision.')
plt.figure(figsize=(10, 4))
plt.plot(time_no, dist_no_hist * 100, label='midpoint fixed auxiliary link', color='blue')
plt.plot(time_no, dglob_no_hist * 100, label='global distance', color='green', ls='--')
plt.axhline(0, color='red', ls='--', lw=1.5, label='contact')
plt.axhline(activation_distance * 100, color='orange', ls=':', lw=1.2, label='activation distance')
plt.axvline(time_no[k_min], color='gray', ls=':', lw=1)
plt.xlabel('time [s]')
plt.ylabel('distance [cm]')
plt.title('distance manipulator-obstacle over time — without secondary task')
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()


In [ ]:
print('\n=== Plot configuration critical without secondary task ===')
q_crit = Q_no[k_min, :]
t_crit = time_no[k_min]
segments_crit = getAllRobotPoints(q_crit)
dists_crit = np.array([pointSegmentDistance(obstacle_center, a, b) - obstacle_radius for a, b in segments_crit])
seg_crit = int(np.argmin(dists_crit))
a_crit, b_crit = segments_crit[seg_crit]
p_close, t_seg, d_center = pointSegmentClosestPoint(obstacle_center, a_crit, b_crit)
print(f'Critical instant: t = {t_crit:.2f} s')
print(f'segment critical: {seg_crit}')
print(f'distance signed segment critical-sphere = {dists_crit[seg_crit] * 100:.3f} cm')
print(f'Closest point on segment = {np.round(p_close, 4)}')
plt.figure(figsize=(8, 8))
circle = plt.Circle((obstacle_center[0], obstacle_center[1]), obstacle_radius, color='red', alpha=0.25, label='obstacle')
plt.gca().add_patch(circle)
activation_circle = plt.Circle((obstacle_center[0], obstacle_center[1]), activation_distance, color='orange', alpha=0.15, fill=False, linestyle='--', linewidth=2, label='activation distance')
plt.gca().add_patch(activation_circle)
for i, (a, b) in enumerate(segments_crit):
    if i == seg_crit:
        plt.plot([a[0], b[0]], [a[1], b[1]], color='red', linewidth=4, label='segment critical')
    else:
        plt.plot([a[0], b[0]], [a[1], b[1]], color='black', linewidth=2)
    plt.plot(a[0], a[1], 'ko', markersize=4)
    plt.plot(b[0], b[1], 'ko', markersize=4)
plt.plot(p_close[0], p_close[1], 'bo', markersize=8, label='closest point')
plt.plot(obstacle_center[0], obstacle_center[1], 'rx', markersize=10, label='centre obstacle')
plt.plot([A[0], B[0], C[0]], [A[1], B[1], C[1]], 'k--', linewidth=1.5, alpha=0.6, label='trajectory A-B-C')
plt.plot(A[0], A[1], 'go', markersize=8, label='A')
plt.plot(B[0], B[1], 'mo', markersize=8, label='B')
plt.plot(C[0], C[1], 'co', markersize=8, label='C')
plt.xlabel('X [m]')
plt.ylabel('Y [m]')
plt.title(f'configuration critical without secondary task in the plane XY — t = {t_crit:.2f} s')
plt.axis('equal')
plt.grid()
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


## Null-space redundancy resolution
Add (I − pinv(J)J) @ (ka × gradient(distance)) when local clearance is below 0.06 m. This is an instantaneous kinematic separation before clipping; joint saturation and finite steps can disturb the primary task.


In [ ]:
def inverseKinematicsRedundant(q, xd, xdot_d, kgain_val=1.0, ka_val=0.0):
    J = Jacobian(q)
    x = getEndEffector3D(q)
    err = xd - x
    K = kgain_val * np.eye(3)
    J_pinv = np.linalg.pinv(J)
    qdot_task = np.matmul(J_pinv, xdot_d + np.matmul(K, err))
    d_obs = obstacleDistance(q)
    if d_obs < activation_distance:
        qdot0 = ka_val * gradObstacleDistance(q)
    else:
        qdot0 = np.zeros(4)
    projector = np.eye(4) - np.matmul(J_pinv, J)
    qdot = qdot_task + np.matmul(projector, qdot0)
    qdot = saturateQdot(qdot)
    return (qdot, err, d_obs)

def simulateTrajectoryFunctional(kgain_val=10.0, ka_val=1.5):
    time = np.arange(t0_traj, t2_traj + dt, dt)
    N = len(time)
    Q = np.zeros((N, 4))
    Pd = np.zeros((N, 3))
    Preal = np.zeros((N, 3))
    error = np.zeros(N)
    dist = np.zeros(N)
    Q[0, :] = q0
    Pd[0, :] = A
    Preal[0, :] = getEndEffector3D(q0)
    error[0] = np.linalg.norm(A - Preal[0, :])
    dist[0] = obstacleDistance(q0)
    for i in range(1, N):
        t = time[i]
        pd, pdot = desiredTrajectory(t)
        qdot, err, d_obs = inverseKinematicsRedundant(Q[i - 1, :], pd, pdot, kgain_val=kgain_val, ka_val=ka_val)
        Q[i, :] = saturateJoints(Q[i - 1, :] + qdot * dt)
        Pd[i, :] = pd
        Preal[i, :] = getEndEffector3D(Q[i, :])
        error[i] = np.linalg.norm(err)
        dist[i] = obstacleDistance(Q[i, :])
    return (time, Q, Pd, Preal, error, dist)


## Symbolic gradient verification
SymPy differentiates the same midpoint distance used by the analytic Jacobian gradient.


In [ ]:
q1_s, q2_s, q3_s, q4_s = sp.symbols('q1 q2 q3 q4', real=True)
l1_s = sp.Rational(8, 100)
l4_s = sp.Rational(85, 1000)
d3_s = sp.Rational(45, 1000)
d3__s = sp.Rational(11, 100)
alpha1_s = -sp.pi / 2
alpha2_s = sp.pi / 2
alpha3_s = -sp.pi / 2
alpha3__s = sp.Integer(0)
alpha4_s = sp.Integer(0)

def DH_sym(d, a, alpha, theta):
    return sp.Matrix([[sp.cos(theta), -sp.sin(theta) * sp.cos(alpha), sp.sin(theta) * sp.sin(alpha), a * sp.cos(theta)], [sp.sin(theta), sp.cos(theta) * sp.cos(alpha), -sp.cos(theta) * sp.sin(alpha), a * sp.sin(theta)], [sp.Integer(0), sp.sin(alpha), sp.cos(alpha), d], [sp.Integer(0), sp.Integer(0), sp.Integer(0), sp.Integer(1)]])
print('symbols and DH parameters defined.')


In [ ]:
print('computing forward kinematics symbolic...')
Tb0_s = sp.Matrix([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, sp.Rational(14, 100)], [0, 0, 0, 1]])
T01_s = DH_sym(sp.Integer(0), l1_s, alpha1_s, q1_s)
T12_s = DH_sym(sp.Integer(0), sp.Integer(0), alpha2_s, q2_s)
T23_s = DH_sym(d3_s, sp.Integer(0), alpha3_s, q3_s)
T23__s = DH_sym(d3__s, sp.Integer(0), alpha3__s, sp.Integer(0))
T34_s = DH_sym(sp.Integer(0), l4_s, alpha4_s, q4_s)
Tb1_s = Tb0_s * T01_s
Tb2_s = Tb1_s * T12_s
Tb3_s = Tb2_s * T23_s
Tb3f_s = Tb3_s * T23__s
Tb4_s = Tb3f_s * T34_s
p3_sym = Tb3_s[:3, 3]
p3f_sym = Tb3f_s[:3, 3]
pmid_sym = (p3_sym + p3f_sym) / 2
print('forward kinematics symbolic computed.')
print(f'pmid_sym: vector of {len(pmid_sym)} components in q1, q2, q3')


In [ ]:
print('computing gradient symbolic')
cx_s = sp.Float(obstacle_center[0])
cy_s = sp.Float(obstacle_center[1])
cz_s = sp.Float(obstacle_center[2])
r_s = sp.Float(obstacle_radius)
delta_sym = pmid_sym - sp.Matrix([cx_s, cy_s, cz_s])
norm_sym = sp.sqrt(delta_sym.dot(delta_sym))
d_sym = norm_sym - r_s
print('  ∂d/∂q1...')
dd_dq1 = sp.diff(d_sym, q1_s)
print('  ∂d/∂q2...')
dd_dq2 = sp.diff(d_sym, q2_s)
print('  ∂d/∂q3...')
dd_dq3 = sp.diff(d_sym, q3_s)
print('  ∂d/∂q4  (expected zero: q4 does not affect pmid)...')
dd_dq4 = sp.diff(d_sym, q4_s)
grad_sym = sp.Matrix([dd_dq1, dd_dq2, dd_dq3, dd_dq4])
print('\ngradient symbolic computed.')
print(f'∂d/∂q4 simplified = {sp.simplify(dd_dq4)}')
print('  → SymPy confirms q4 does not affect the fixed-link midpoint.')


In [ ]:
grad_func_sympy = sp.lambdify([q1_s, q2_s, q3_s, q4_s], grad_sym, modules='numpy')

def gradObstacleDistance_sympy(q):
    result = grad_func_sympy(q[0], q[1], q[2], q[3])
    return np.array(result, dtype=float).flatten()
print('=== Verification: gradient analytic vs SymPy ===')
q_test_sym = np.array([0.3, 0.1, -0.2, 0.5])
grad_an = gradObstacleDistance(q_test_sym)
grad_sy = gradObstacleDistance_sympy(q_test_sym)
err_max = np.max(np.abs(grad_an - grad_sy))
print(f'q_test = {np.round(q_test_sym, 3)}')
print(f'\ngradient analytic : {np.round(grad_an, 8)}')
print(f'gradient SymPy     : {np.round(grad_sy, 8)}')
print(f'\nmaximum error      : {err_max:.2e}')
if err_max < 1e-08:
    print('✓ Both gradients match numerically.')
else:
    print('⚠ Significant discrepancy; check parameters.')


In [ ]:
def inverseKinematicsRedundant_sympy(q, xd, xdot_d, kgain_val=1.0, ka_val=0.0):
    J = Jacobian(q)
    x = getEndEffector3D(q)
    err = xd - x
    K = kgain_val * np.eye(3)
    J_pinv = np.linalg.pinv(J)
    qdot_task = J_pinv @ (xdot_d + K @ err)
    d_obs = obstacleDistance(q)
    if d_obs < activation_distance:
        qdot0 = ka_val * gradObstacleDistance_sympy(q)
    else:
        qdot0 = np.zeros(4)
    projector = np.eye(4) - J_pinv @ J
    qdot = qdot_task + projector @ qdot0
    qdot = saturateQdot(qdot)
    return (qdot, err, d_obs)

def simulateTrajectoryFunctional_sympy(kgain_val=10.0, ka_val=1.5):
    time = np.arange(t0_traj, t2_traj + dt, dt)
    N = len(time)
    Q = np.zeros((N, 4))
    Pd = np.zeros((N, 3))
    Preal = np.zeros((N, 3))
    error = np.zeros(N)
    dist = np.zeros(N)
    Q[0, :] = q0
    Pd[0, :] = A
    Preal[0, :] = getEndEffector3D(q0)
    error[0] = np.linalg.norm(A - Preal[0, :])
    dist[0] = obstacleDistance(q0)
    for i in range(1, N):
        t = time[i]
        pd, pdot = desiredTrajectory(t)
        qdot, err, d_obs = inverseKinematicsRedundant_sympy(Q[i - 1, :], pd, pdot, kgain_val=kgain_val, ka_val=ka_val)
        Q[i, :] = saturateJoints(Q[i - 1, :] + qdot * dt)
        Pd[i, :] = pd
        Preal[i, :] = getEndEffector3D(Q[i, :])
        error[i] = np.linalg.norm(err)
        dist[i] = obstacleDistance(Q[i, :])
    return (time, Q, Pd, Preal, error, dist)


## Gain search
Preserve the implemented selection rule: among tracking-valid candidates prefer gains ≤10, then minimise mean error; this is not simply the smallest admissible gain. For ka, maximise minimum local distance subject to clearance and joint-margin thresholds, with original fallback rules.


In [ ]:
def minimumJointMarginDeg(Q):
    margin = np.minimum(Q - q_min, q_max - Q)
    return float(np.min(np.degrees(margin)))

def jointLimitReport(Q, tol_deg=0.001):
    Q_deg = np.degrees(Q)
    qmin_deg = np.degrees(q_min)
    qmax_deg = np.degrees(q_max)
    margin = np.minimum(Q_deg - qmin_deg, qmax_deg - Q_deg)
    min_marg = np.min(margin, axis=0)
    crit_q = int(np.argmin(min_marg)) + 1
    lower_hits = np.sum(np.abs(Q_deg - qmin_deg) <= tol_deg, axis=0)
    upper_hits = np.sum(np.abs(Q_deg - qmax_deg) <= tol_deg, axis=0)
    return {'global_min_margin_deg': float(np.min(min_marg)), 'critical_joint': crit_q, 'q_min_reached_deg': np.min(Q_deg, axis=0), 'q_max_reached_deg': np.max(Q_deg, axis=0), 'lower_hits': lower_hits, 'upper_hits': upper_hits}

def evaluateGainTrial(kgain_val, ka_val):
    time_t, Q_t, Pd_t, Preal_t, err_t, dist_t = simulateTrajectoryFunctional(kgain_val, ka_val)
    rep = jointLimitReport(Q_t)
    return {'kgain': kgain_val, 'ka': ka_val, 'min_dist_cm': float(np.min(dist_t) * 100), 'mean_err_mm': float(np.mean(err_t) * 1000), 'max_err_mm': float(np.max(err_t) * 1000), 'min_joint_margin_deg': rep['global_min_margin_deg'], 'critical_joint': rep['critical_joint'], 'q_min_reached_deg': rep['q_min_reached_deg'], 'q_max_reached_deg': rep['q_max_reached_deg'], 'lower_hits': rep['lower_hits'], 'upper_hits': rep['upper_hits']}


In [ ]:
min_dist_required_cm = 0.6
min_joint_margin_deg = 2.0
mean_err_thresh_mm = 0.35
max_err_thresh_mm = 0.6
kgain_values = np.array([0.5, 1.0, 2.0, 4.0, 6.0, 8.0, 10.0, 15.0, 20.0, 30.0, 40.0, 50.0, 60.0])
ka_values = np.array([0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0, 12.0, 15.0, 20.0])
print('=== step 1: tuning of kgain (ka = 0) ===')
kgain_results = []
for kg in kgain_values:
    r = evaluateGainTrial(kg, 0.0)
    kgain_results.append(r)
    print(f"kgain={kg:5.1f} | mean_err={r['mean_err_mm']:.3f} mm | max_err={r['max_err_mm']:.3f} mm | d_min={r['min_dist_cm']:.3f} cm")
valid_kg = [r for r in kgain_results if r['mean_err_mm'] <= mean_err_thresh_mm and r['max_err_mm'] <= max_err_thresh_mm]
if valid_kg:
    best_kgain = max(valid_kg, key=lambda r: (r['kgain'] <= 10, -r['mean_err_mm']))['kgain']
else:
    best_kgain = min(kgain_results, key=lambda r: r['mean_err_mm'])['kgain']
print(f'\n→ kgain selected: {best_kgain}')


In [ ]:
print(f'\n=== step 2: tuning of ka (kgain = {best_kgain}) ===')
ka_results = []
for ka in ka_values:
    r = evaluateGainTrial(best_kgain, ka)
    ka_results.append(r)
    safe = ('✓' if r['min_dist_cm'] >= min_dist_required_cm else '✗') + ('✓' if r['min_joint_margin_deg'] >= min_joint_margin_deg else '✗')
    print(f"ka={ka:5.2f} | d_min={r['min_dist_cm']:+.3f} cm | err={r['mean_err_mm']:.3f} mm | margin={r['min_joint_margin_deg']:.2f}° | crit_q=q{r['critical_joint']} | {safe}")
safe_both = [r for r in ka_results if r['min_dist_cm'] >= min_dist_required_cm and r['min_joint_margin_deg'] >= min_joint_margin_deg]
safe_distance = [r for r in ka_results if r['min_dist_cm'] >= min_dist_required_cm]
if safe_both:
    best_result = max(safe_both, key=lambda r: (r['min_dist_cm'], -r['ka']))
elif safe_distance:
    best_result = max(safe_distance, key=lambda r: (r['min_dist_cm'], -r['ka']))
else:
    best_result = max(ka_results, key=lambda r: (r['min_dist_cm'], -r['ka']))
best_ka = best_result['ka']
best_result_distance_only = max(ka_results, key=lambda r: (r['min_dist_cm'], -r['ka']))
best_ka_distance_only = best_result_distance_only['ka']
print(f'\n→ ka selected: {best_ka}')
print(f"  minimum distance with secondary task: {best_result['min_dist_cm']:.3f} cm")
print(f"  mean error tracking:          {best_result['mean_err_mm']:.3f} mm")
print(f"  joint margin minimum:      {best_result['min_joint_margin_deg']:.2f}°")
print(f"  joint critical:                 q{best_result['critical_joint']}")
print(f'\n→ ka selected using distance only: {best_ka_distance_only}')
print(f"  minimum distance distance only:        {best_result_distance_only['min_dist_cm']:.3f} cm")
print(f"  joint margin minimum:            {best_result_distance_only['min_joint_margin_deg']:.2f}°")
print(f"  joint critical:                       q{best_result_distance_only['critical_joint']}")


In [ ]:
print('=== Obstacle clearance curves for all ka values ===')
plt.figure(figsize=(12, 6))
for ka in ka_values:
    time_k, Q_k, Pd_k, Preal_k, error_k, dist_k = simulateTrajectoryFunctional(kgain_val=best_kgain, ka_val=ka)
    label = f'ka = {ka:.2f}'
    if ka == best_ka:
        plt.plot(time_k, dist_k * 100, linewidth=3.0, color='blue', label=label + '  (selected)')
    elif ka == best_ka_distance_only:
        plt.plot(time_k, dist_k * 100, linewidth=2.8, linestyle='--', color='purple', label=label + '  (distance only)')
    else:
        plt.plot(time_k, dist_k * 100, linewidth=1.2, alpha=0.55, label=label)
plt.axhline(0, color='red', linestyle='--', linewidth=1.4, label='contact')
plt.axhline(min_dist_required_cm, color='black', linestyle=':', linewidth=1.5, label=f'threshold distance = {min_dist_required_cm:.2f} cm')
plt.axhline(activation_distance * 100, color='gray', linestyle='-.', linewidth=1.2, label=f'activation distance = {activation_distance * 100:.1f} cm')
plt.xlabel('time [s]')
plt.ylabel('distance midpoint fixed auxiliary link - obstacle [cm]')
plt.title('distance from obstacle over time for all ka values')
plt.grid(True, alpha=0.4)
plt.legend(fontsize=7, loc='center left', bbox_to_anchor=(1.02, 0.5), borderaxespad=0)
plt.tight_layout()
plt.show()
print('\n=== minimum distance as a function of ka ===')
print(f"{'ka':>8} | {'minimum distance [cm]':>22} | {'mean error [mm]':>18} | {'joint margin [deg]':>20}")
print('-' * 80)
for r in ka_results:
    marker = ''
    if r['ka'] == best_ka:
        marker = '  <-- selected'
    elif r['ka'] == best_ka_distance_only:
        marker = '  <-- distance only'
    print(f"{r['ka']:>8.2f} | {r['min_dist_cm']:>22.3f} | {r['mean_err_mm']:>18.3f} | {r['min_joint_margin_deg']:>20.2f}{marker}")


## Final simulation comparison
Compare primary tracking alone with the selected null-space task. Preserve numerical results and distinguish them from physical tests.


In [ ]:
print(f'final gains: kgain = {best_kgain},  ka = {best_ka}')
time_no, Q_no, Pd_no, Preal_no, error_no = simulateTrajectoryNoFunctional(kgain_val=best_kgain)
dist_no = np.array([obstacleDistance(q) for q in Q_no])
time_fun, Q_fun, Pd_fun, Preal_fun, error_fun, dist_fun = simulateTrajectoryFunctional(kgain_val=best_kgain, ka_val=best_ka)
dmin_no = np.min(dist_no) * 100
dmin_fun = np.min(dist_fun) * 100
print(f'\nminimum distance without secondary task: {dmin_no:+.3f} cm')
print(f'minimum distance with  secondary task: {dmin_fun:+.3f} cm')
print(f'improvement:                   {dmin_fun - dmin_no:+.3f} cm')
if dmin_no < 0:
    print('\n✓ without secondary task: slight collision in this test scenario.')
if dmin_fun >= 0:
    print('✓ with secondary task: non-negative local clearance in this scenario.')
else:
    print('⚠ with secondary task: negative local clearance; review gains and geometry.')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
comp_labels = ['x [m]', 'y [m]', 'z [m]']
for i, (label, ax) in enumerate(zip(comp_labels, axes.flat)):
    ax.plot(time_no, Pd_no[:, i], 'r--', lw=1.5, label='desired')
    ax.plot(time_no, Preal_no[:, i], color='gray', lw=2, label='without secondary task')
    ax.plot(time_fun, Preal_fun[:, i], color='blue', lw=2, label='with secondary task')
    ax.axvline(t1_traj, color='gray', ls=':', lw=1)
    ax.set_ylabel(label)
    ax.set_xlabel('time [s]')
    ax.legend(fontsize=8)
    ax.grid()
axes[1, 1].plot(time_no, dist_no * 100, color='gray', lw=2, label='without secondary task')
axes[1, 1].plot(time_fun, dist_fun * 100, color='blue', lw=2, label='with secondary task')
axes[1, 1].axhline(0, color='red', ls='--', lw=1.5, label='contact')
axes[1, 1].axhline(3.0, color='orange', ls='--', lw=1.2, label='3 cm')
axes[1, 1].set_ylabel('distance [cm]')
axes[1, 1].set_xlabel('time [s]')
axes[1, 1].set_title('Obstacle clearance')
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid()
plt.suptitle(f'Tracking with and without the secondary task  (kgain={best_kgain}, ka={best_ka})')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
q_min_deg = np.degrees(q_min)
q_max_deg = np.degrees(q_max)
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']
for i, ax in enumerate(axes.flat):
    ax.plot(time_no, np.degrees(Q_no[:, i]), '--', color=colors[i], lw=1.5, label='without secondary task', alpha=0.8)
    ax.plot(time_fun, np.degrees(Q_fun[:, i]), '-', color=colors[i], lw=2.5, label='with secondary task')
    ax.axhline(q_min_deg[i], ls=':', lw=1.2, color='black', alpha=0.7)
    ax.axhline(q_max_deg[i], ls=':', lw=1.2, color='black', alpha=0.7)
    ax.axvline(t1_traj, color='gray', ls=':', lw=1)
    ax.set_title(f'q{i + 1}  [limits: {q_min_deg[i]:.0f}°, {q_max_deg[i]:.0f}°]')
    ax.set_xlabel('time [s]')
    ax.set_ylabel('angle [deg]')
    ax.legend(fontsize=8)
    ax.grid()
plt.suptitle('comparison trajectory joints — without vs with secondary task')
plt.tight_layout()
plt.show()


In [ ]:
print('=' * 55)
print('FINAL SUMMARY')
print('=' * 55)
print(f'kgain = {best_kgain},  ka = {best_ka}')
print(f'obstacle: centre={obstacle_center}, radius={obstacle_radius} m')
print()
print(f"{'Metric':<35} {'without':>10} {'with':>10}")
print('-' * 55)
print(f"{'minimum distance [cm]':<35} {dmin_no:>+10.3f} {dmin_fun:>+10.3f}")
print(f"{'mean error tracking [mm]':<35} {np.mean(error_no) * 1000:>10.3f} {np.mean(error_fun) * 1000:>10.3f}")
print(f"{'maximum error tracking [mm]':<35} {np.max(error_no) * 1000:>10.3f} {np.max(error_fun) * 1000:>10.3f}")
print(f"{'q2 max reached [deg]':<35} {np.max(np.degrees(Q_no[:, 1])):>10.2f} {np.max(np.degrees(Q_fun[:, 1])):>10.2f}")
print(f"{'q2 min reached [deg]':<35} {np.min(np.degrees(Q_no[:, 1])):>10.2f} {np.min(np.degrees(Q_fun[:, 1])):>10.2f}")
print('=' * 55)


## Simulation using the symbolic gradient
Use the same selected gains and compare analytic and symbolic distance gradients.


In [ ]:
print(f'simulation SymPy: kgain = {best_kgain},  ka = {best_ka}')
time_sym, Q_sym, Pd_sym, Preal_sym, error_sym, dist_sym = simulateTrajectoryFunctional_sympy(kgain_val=best_kgain, ka_val=best_ka)
print('\n=== comparison: gradient analytic vs SymPy ===')
print(f"{'Metric':<35} {'analytic':>12} {'SymPy':>12} {'Diff':>12}")
print('-' * 73)
rows = [('distance min [cm]', np.min(dist_fun) * 100, np.min(dist_sym) * 100), ('mean error tracking [mm]', np.mean(error_fun) * 1000, np.mean(error_sym) * 1000), ('maximum error tracking [mm]', np.max(error_fun) * 1000, np.max(error_sym) * 1000), ('q2 max reached [deg]', np.max(np.degrees(Q_fun[:, 1])), np.max(np.degrees(Q_sym[:, 1]))), ('q2 min reached [deg]', np.min(np.degrees(Q_fun[:, 1])), np.min(np.degrees(Q_sym[:, 1])))]
for label, v_an, v_sy in rows:
    print(f'  {label:<33} {v_an:>12.4f} {v_sy:>12.4f} {abs(v_an - v_sy):>12.2e}')
print('\n→ Numerically matching results: both methods compute')
print('  the same gradient using different approaches.')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for i, (label, ax) in enumerate(zip(['x [m]', 'y [m]', 'z [m]'], axes.flat)):
    ax.plot(time_fun, Pd_fun[:, i], 'r--', lw=1.5, label='desired')
    ax.plot(time_fun, Preal_fun[:, i], color='blue', lw=2, label='analytic')
    ax.plot(time_sym, Preal_sym[:, i], color='orange', lw=2, ls='--', label='SymPy')
    ax.axvline(t1_traj, color='gray', ls=':', lw=1)
    ax.set_ylabel(label)
    ax.set_xlabel('time [s]')
    ax.legend(fontsize=8)
    ax.grid()
axes[1, 1].plot(time_fun, dist_fun * 100, color='blue', lw=2, label='analytic')
axes[1, 1].plot(time_sym, dist_sym * 100, color='orange', lw=2, ls='--', label='SymPy')
axes[1, 1].axhline(0, color='red', ls='--', lw=1.5, label='contact')
axes[1, 1].set_ylabel('distance [cm]')
axes[1, 1].set_xlabel('time [s]')
axes[1, 1].set_title('Obstacle clearance')
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid()
plt.suptitle(f'comparison: gradient analytic vs SymPy  (kgain={best_kgain}, ka={best_ka})\nthe curves overlap — the two approaches are equivalent')
plt.tight_layout()
plt.show()
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for i, ax in enumerate(axes.flat):
    ax.plot(time_fun, np.degrees(Q_fun[:, i]), color='blue', lw=2.5, label='analytic')
    ax.plot(time_sym, np.degrees(Q_sym[:, i]), color='orange', lw=2, ls='--', label='SymPy')
    ax.axhline(np.degrees(q_min[i]), ls=':', lw=1.2, color='black', alpha=0.6)
    ax.axhline(np.degrees(q_max[i]), ls=':', lw=1.2, color='black', alpha=0.6)
    ax.axvline(t1_traj, color='gray', ls=':', lw=1)
    ax.set_title(f'q{i + 1}  [{np.degrees(q_min[i]):.0f}°, {np.degrees(q_max[i]):.0f}°]')
    ax.set_xlabel('time [s]')
    ax.set_ylabel('angle [deg]')
    ax.legend(fontsize=8)
    ax.grid()
plt.suptitle('comparison joints: gradient analytic vs SymPy')
plt.tight_layout()
plt.show()


## Global clearance analysis
Evaluate every line segment, not only the chosen midpoint. This is a diagnostic; the controller optimises local midpoint distance.


In [ ]:
dist_global_no = np.array([globalObstacleDistance(q) for q in Q_no])
dist_global_fun = np.array([globalObstacleDistance(q) for q in Q_fun])
print('\n=== Global manipulator-obstacle clearance analysis ===')
print(f'minimum distance global without secondary task [cm] = {np.min(dist_global_no) * 100:.3f}')
print(f'minimum distance global with  secondary task [cm] = {np.min(dist_global_fun) * 100:.3f}')
print(f'improvement global [cm] = {(np.min(dist_global_fun) - np.min(dist_global_no)) * 100:.3f}')
print('\n=== comparison with distance of the fixed auxiliary link ===')
print(f'minimum distance local without secondary task [cm] = {np.min(dist_no) * 100:.3f}')
print(f'minimum distance local with  secondary task [cm] = {np.min(dist_fun) * 100:.3f}')
print(f'improvement local [cm] = {(np.min(dist_fun) - np.min(dist_no)) * 100:.3f}')


### Interpretation
The original selected run improves global centreline clearance as well as local midpoint clearance. This single scenario does not guarantee collision avoidance for arbitrary obstacles or physical link volumes.


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(time_no, dist_no * 100, color='gray', linewidth=2, label='local distance - without secondary task')
plt.plot(time_fun, dist_fun * 100, color='blue', linewidth=2, label='local distance - with secondary task')
plt.plot(time_no, dist_global_no * 100, '--', color='black', linewidth=2, label='global distance - without secondary task')
plt.plot(time_fun, dist_global_fun * 100, '--', color='green', linewidth=2, label='global distance - with secondary task')
plt.axhline(0, color='red', linestyle='--', linewidth=1.3, label='contact')
plt.xlabel('time [s]')
plt.ylabel('distance [cm]')
plt.title('Local and global clearance comparison')
plt.grid()
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
seg_labels = ['base → p0', 'p0 → p1  (link 1)', 'p1 → p2  (link 2)', 'p2 → p3  (link 3)', 'p3 → p3_f (fixed auxiliary link)', 'p3_f → p4 (link 4)']
seg_crit_hist = np.zeros(len(Q_no), dtype=int)
for k in range(len(Q_no)):
    segs = getAllRobotPoints(Q_no[k, :])
    dists = [pointSegmentDistance(obstacle_center, a, b) - obstacle_radius for a, b in segs]
    seg_crit_hist[k] = int(np.argmin(dists))
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
colors_seg = ['tab:gray', 'tab:blue', 'tab:green', 'tab:orange', 'tab:red', 'tab:purple']
dist_per_seg = np.zeros((len(Q_no), 6))
for k in range(len(Q_no)):
    segs = getAllRobotPoints(Q_no[k, :])
    for s_idx, (a, b) in enumerate(segs):
        dist_per_seg[k, s_idx] = pointSegmentDistance(obstacle_center, a, b) - obstacle_radius
for s_idx in range(6):
    axes[0].plot(time_no, dist_per_seg[:, s_idx] * 100, color=colors_seg[s_idx], lw=1.5, label=seg_labels[s_idx])
axes[0].axhline(0, color='red', ls='--', lw=1.5, label='contact')
axes[0].set_ylabel('distance [cm]')
axes[0].set_title('distance of each segment from the obstacle over time')
axes[0].legend(fontsize=8, ncol=2)
axes[0].grid()
axes[1].step(time_no, seg_crit_hist, where='mid', color='black', lw=2)
axes[1].set_yticks(range(6))
axes[1].set_yticklabels(seg_labels, fontsize=8)
axes[1].set_xlabel('time [s]')
axes[1].set_ylabel('segment critical')
axes[1].set_title('segment of the manipulator closest to the obstacle over time')
axes[1].grid(axis='x')
plt.tight_layout()
plt.show()
print('\nSample count by critical segment:')
for s_idx in range(6):
    count = np.sum(seg_crit_hist == s_idx)
    pct = count / len(seg_crit_hist) * 100
    print(f'  {seg_labels[s_idx]:30s}: {count:4d} samples ({pct:.1f}%)')


## Aggressive distance-only selection
Maximising clearance without the joint-margin requirement selects ka=3 in the saved run, saturating joint 2 and increasing tracking error.


In [ ]:
print('\n=== ka alternative: maximum obstacle distance ===')
best_ka_distance_only_result = best_result_distance_only
best_ka_distance_only = best_ka_distance_only_result['ka']
print(f'ka selected using distance only = {best_ka_distance_only:.2f}')
print(f"minimum distance = {best_ka_distance_only_result['min_dist_cm']:.3f} cm")
print(f"mean error = {best_ka_distance_only_result['mean_err_mm']:.3f} mm")
print(f"maximum error = {best_ka_distance_only_result['max_err_mm']:.3f} mm")
print(f"joint margin = {best_ka_distance_only_result['min_joint_margin_deg']:.3f} deg")
print(f"critical joint = q{best_ka_distance_only_result['critical_joint']}")
print('\nSaturation report for distance-only case:')
print('Maximum reached joint angles [deg] = {}'.format(np.round(best_ka_distance_only_result['q_max_reached_deg'], 3)))
print('Minimum reached joint angles [deg] = {}'.format(np.round(best_ka_distance_only_result['q_min_reached_deg'], 3)))
print('samples at lower limit = {}'.format(best_ka_distance_only_result['lower_hits']))
print('samples at upper limit = {}'.format(best_ka_distance_only_result['upper_hits']))


In [ ]:
print('\n=== simulation for the aggressive case ===')
time_alt, Q_alt, Pd_alt, Preal_alt, err_alt, dist_alt = simulateTrajectoryFunctional(best_kgain, best_ka_distance_only)
print(f'kgain = {best_kgain:.2f}')
print(f'aggressive ka = {best_ka_distance_only:.2f}')
print(f'minimum distance [cm] = {np.min(dist_alt) * 100:.3f}')
print(f'mean error [mm] = {np.mean(err_alt) * 1000:.3f}')
print(f'maximum error [mm] = {np.max(err_alt) * 1000:.3f}')
print(f'minimum joint margin [deg] = {minimumJointMarginDeg(Q_alt):.3f}')


### Joint-margin tradeoff
The selected ka=1.5 retains joint margin; ka=3 increases clearance while driving joint 2 to its upper limit.


In [ ]:
plt.figure(figsize=(11, 7))
q_min_deg = np.rad2deg(q_min)
q_max_deg = np.rad2deg(q_max)
for i in range(4):
    plt.subplot(2, 2, i + 1)
    plt.plot(time_fun, np.rad2deg(Q_fun[:, i]), linewidth=2, label=f'q{i + 1} selected')
    plt.plot(time_alt, np.rad2deg(Q_alt[:, i]), '--', linewidth=2, label=f'q{i + 1} distance only')
    plt.axhline(q_min_deg[i], linestyle=':', linewidth=1.5, color='red', label='Lower limit' if i == 0 else None)
    plt.axhline(q_max_deg[i], linestyle=':', linewidth=1.5, color='red', label='Upper limit' if i == 0 else None)
    plt.xlabel('time [s]')
    plt.ylabel('angle [deg]')
    plt.title(f'Joint trajectory q{i + 1}')
    plt.grid()
    plt.legend(fontsize=8)
plt.tight_layout()
plt.show()
plt.figure(figsize=(10, 5))
plt.plot(time_fun, np.rad2deg(Q_fun[:, 1]), linewidth=2.5, label=f'q2 selected (ka = {best_ka:.2f})')
plt.plot(time_alt, np.rad2deg(Q_alt[:, 1]), '--', linewidth=2.5, label=f'q2 distance only (ka = {best_ka_distance_only:.2f})')
plt.axhline(np.rad2deg(q_min[1]), linestyle=':', linewidth=1.8, color='red', label='q2 min')
plt.axhline(np.rad2deg(q_max[1]), linestyle=':', linewidth=1.8, color='red', label='q2 max')
plt.xlabel('time [s]')
plt.ylabel('q2 [deg]')
plt.title('Joint 2 saturation comparison')
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()


## Tracking under saturation
Compare the preserved tracking-error convention for selected and aggressive gains.


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(time_fun, error_fun * 1000, linewidth=2.5, label=f'final error (ka = {best_ka:.2f})')
plt.plot(time_alt, err_alt * 1000, '--', linewidth=2.5, label=f'error distance only (ka = {best_ka_distance_only:.2f})')
plt.xlabel('time [s]')
plt.ylabel('tracking error [mm]')
plt.title('Trajectory tracking-error comparison')
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()
print('\n=== comparison tracking errors ===')
print(f'mean error selected [mm] = {np.mean(error_fun) * 1000:.3f}')
print(f'mean error distance only [mm] = {np.mean(err_alt) * 1000:.3f}')
print(f'maximum error selected [mm] = {np.max(error_fun) * 1000:.3f}')
print(f'maximum error distance only [mm] = {np.max(err_alt) * 1000:.3f}')


## Verified simulation results

Historical and current software-check values (rounded): kgain=10, ka=1.5; local clearance 0.649 cm, global clearance 0.636 cm, mean logged tracking error 0.007 mm, maximum 0.014 mm, q2 maximum 29.16 degrees. Aggressive ka=3: local clearance 1.172 cm, mean error 0.237 mm and zero joint margin. No physical accuracy is implied.

![Verified project result](../assets/simulation/tracking_and_avoidance_0.png)

![Verified project result](../assets/simulation/joint_saturation_0.png)